<h1 style="font-size:40px;"> Dataframe: A fondo </h1>

![](img/lrg.jpg)

&nbsp;   


Vamos a ver con más detalle qué podemos hacer con la API `DataFrame` de spark. Dos buenas referencias son la [documentación oficial](https://spark.apache.org/docs/latest/sql-programming-guide.html) y el libro [*Learning Spark*](http://shop.oreilly.com/product/0636920028512.do)


Empezamos iniciando la sesión:

In [1]:
import os

import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark import SparkConf
from pyspark.sql import Row, SparkSession

In [2]:
conf = (
    SparkConf()
    .setAppName("[ICAI] DataFrame: A fondo")
    .set("spark.jars", "/var/lib/sqoop/mysql-connector-java-5.1.44-bin.jar")
)

In [3]:
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


# RDD vs Dataframe

En el anterior sesión vimos que trabajar con `DataFrames` es en general más rápido que trabajar con RDD (y  más sobre todo si trabajamos con *Pyspark*, pero ¿Por qué?




![](img/dataframe.png)
<center>
    https://databricks.com/blog/2015/02/17/introducing-dataframes-in-spark-for-large-scale-data-science.html
</center>

Básicamente al trabajar con datos estructurados se puede usar compresores específicos para los tipos normales de una tabla y además se puede saber qué se está haciendo en cada operación y se pude preeveer el resultado final de cada operación, en spark hay dos proyectos que hacen esto posible:

* **Project Tungsten**: https://databricks.com/blog/2015/04/28/project-tungsten-bringing-spark-closer-to-bare-metal.html
* **Spark SQL’s Catalyst Optimizer**: https://databricks.com/blog/2015/04/13/deep-dive-into-spark-sqls-catalyst-optimizer.html

![](img/catalyst.png)

### Trabajando con  RDDs y DFs

Podemos convertir de `RDD` a `DadaFrame` y viceversa con gran facilidad:

In [4]:
lines = spark.sparkContext.textFile("/datos/people.txt")

In [5]:
parts = lines.map(lambda l: l.split(","))

Usaremos `Row` para definir los registros del RDD como estructurados (filas):

In [6]:
people = parts.map(lambda p: Row(name=p[0], age=int(p[1])))

In [7]:
schemaPeople = people.toDF()

In [8]:
schemaPeople.show()

+-------+---+
|   name|age|
+-------+---+
|Michael| 29|
|   Andy| 30|
| Justin| 19|
+-------+---+



In [9]:
teenagers = schemaPeople.filter("age between 13 and 19").select("name")

In [10]:
teenagers.show()

+------+
|  name|
+------+
|Justin|
+------+



Con `.rdd` obtenemos el RDD que hay dentro del DataFrame:

In [11]:
teenNames = teenagers.rdd.map(lambda p: "Name: " + p.name).collect()

In [12]:
for name in teenNames:
    print(name)

Name: Justin


## Funciones

![](img/relation-not-function.gif)

Dentro de la API de `DataFrame` hay multitud de funciones que podemos usar para nuestros análisis. Podemos ver todas ellas en la [documetanción](http://spark.apache.org/docs/latest/api/python/pyspark.sql.html#module-pyspark.sql.functions).

Veremos algunos ejemplos basados en siguiente artículo de Databricks:
https://databricks.com/blog/2015/06/02/statistical-and-mathematical-functions-with-dataframes-in-spark.html

####  1. Random Data Generation

In [13]:
df = spark.range(0, 10)

In [14]:
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [15]:
df = df.select("id", F.rand(seed=10).alias("uniform"), F.randn(seed=27).alias("normal"))

In [16]:
df.show()

+---+-------------------+--------------------+
| id|            uniform|              normal|
+---+-------------------+--------------------+
|  0| 0.1709497137955568| -0.8664700627108758|
|  1| 0.8051143958005459| -0.5970491018333267|
|  2| 0.5775925576589018| 0.18267161219540898|
|  3| 0.9476047869880925| -1.8497305679917546|
|  4|    0.2093704977577|  0.9410417279045351|
|  5|0.03422639313807285| 0.45800664187768786|
|  6|0.11682250456449328|-0.48255190429809436|
|  7| 0.3252227809451943|  1.0749269038930334|
|  8| 0.4358851046481089| -0.9216112248597651|
|  9|0.27831507107951303|  0.5997780578594183|
+---+-------------------+--------------------+



#### 2. Summary and Descriptive Statistics


In [17]:
df.describe().show()

+-------+------------------+-------------------+--------------------+
|summary|                id|            uniform|              normal|
+-------+------------------+-------------------+--------------------+
|  count|                10|                 10|                  10|
|   mean|               4.5|0.39011038063761794|-0.14609879179637328|
| stddev|3.0276503540974917| 0.3016657105926972|  0.9452463698357684|
|    min|                 0|0.03422639313807285| -1.8497305679917546|
|    max|                 9| 0.9476047869880925|  1.0749269038930334|
+-------+------------------+-------------------+--------------------+



In [18]:
df.describe("uniform", "normal").show()

+-------+-------------------+--------------------+
|summary|            uniform|              normal|
+-------+-------------------+--------------------+
|  count|                 10|                  10|
|   mean|0.39011038063761794|-0.14609879179637328|
| stddev| 0.3016657105926972|  0.9452463698357684|
|    min|0.03422639313807285| -1.8497305679917546|
|    max| 0.9476047869880925|  1.0749269038930334|
+-------+-------------------+--------------------+



In [19]:
df.select(F.mean("uniform"), F.min("uniform"), F.max("uniform")).show()

+-------------------+-------------------+------------------+
|       avg(uniform)|       min(uniform)|      max(uniform)|
+-------------------+-------------------+------------------+
|0.39011038063761794|0.03422639313807285|0.9476047869880925|
+-------------------+-------------------+------------------+



En muchas ocasiones, es adecuado renombrar la nueva columna:

In [20]:
df.select(
    F.mean("uniform").alias("media"),
    F.min("uniform").alias("minimo"),
    F.max("uniform").alias("maximo"),
).show()

+-------------------+-------------------+------------------+
|              media|             minimo|            maximo|
+-------------------+-------------------+------------------+
|0.39011038063761794|0.03422639313807285|0.9476047869880925|
+-------------------+-------------------+------------------+



#### 3. Sample covariance and correlation

In [21]:
df = spark.range(0, 10).withColumn("rand1", F.rand(seed=10)).withColumn("rand2", F.rand(seed=27))

In [22]:
df.stat.cov("rand1", "rand2")

-0.015801844353832264

In [23]:
df.stat.cov("id", "id")

9.166666666666666

In [24]:
df.stat.corr("rand1", "rand2")

-0.1662238873855882

In [25]:
df.stat.corr("id", "id")

1.0

#### 4. Cross Tabulation (Contingency Table)

In [26]:
# Create a DataFrame with two columns (name, item)
names = ["Alice", "Bob", "Mike"]
items = ["milk", "bread", "butter", "apples", "oranges"]

In [27]:
pares = [(names[i % 3], items[i % 5]) for i in range(100)]

In [28]:
pares[:5]

[('Alice', 'milk'),
 ('Bob', 'bread'),
 ('Mike', 'butter'),
 ('Alice', 'apples'),
 ('Bob', 'oranges')]

In [29]:
df = spark.createDataFrame(pares, ["name", "item"])

In [30]:
df.show(5)

+-----+-------+
| name|   item|
+-----+-------+
|Alice|   milk|
|  Bob|  bread|
| Mike| butter|
|Alice| apples|
|  Bob|oranges|
+-----+-------+
only showing top 5 rows



In [31]:
df.stat.crosstab("name", "item").show()

+---------+------+-----+------+----+-------+
|name_item|apples|bread|butter|milk|oranges|
+---------+------+-----+------+----+-------+
|      Bob|     6|    7|     7|   6|      7|
|    Alice|     7|    7|     6|   7|      7|
|     Mike|     7|    6|     7|   7|      6|
+---------+------+-----+------+----+-------+



#### 5. Frequent Items

In [32]:
df = spark.createDataFrame(
    [(1, 2, 3) if i % 2 == 0 else (i, 2 * i, i % 4) for i in range(100)],
    ["a", "b", "c"],
)

In [33]:
df.show(5)

+---+---+---+
|  a|  b|  c|
+---+---+---+
|  1|  2|  3|
|  1|  2|  1|
|  1|  2|  3|
|  3|  6|  3|
|  1|  2|  3|
+---+---+---+
only showing top 5 rows



In [34]:
freq = df.stat.freqItems(["a", "b", "c"], 0.4)

In [35]:
freq.show()

+-----------+-----------+-----------+
|a_freqItems|b_freqItems|c_freqItems|
+-----------+-----------+-----------+
|    [1, 99]|   [2, 198]|     [1, 3]|
+-----------+-----------+-----------+



#### 6. Mathematical Functions

In [36]:
df = spark.range(0, 10).withColumn("uniform", F.rand(seed=10) * 3.14)

In [37]:
df.show()

+---+-------------------+
| id|            uniform|
+---+-------------------+
|  0| 0.5367821013180484|
|  1|  2.528059202813714|
|  2|  1.813640631048952|
|  3| 2.9754790311426107|
|  4|  0.657423362959178|
|  5|0.10747087445354876|
|  6|0.36682266433250893|
|  7| 1.0211995321679102|
|  8|  1.368679228595062|
|  9|  0.873909323189671|
+---+-------------------+



In [38]:
(
    df.select(
        "uniform",
        # Convertir en grados
        F.toDegrees("uniform"),
        (
            # cos^2(x)
            F.pow(F.cos(df["uniform"]), 2)
            +
            # sin^2(x)
            F.pow(F.sin(df.uniform), 2)
        ).alias("cos^2 + sin^2"),
    )
).show()

+-------------------+------------------+------------------+
|            uniform|  DEGREES(uniform)|     cos^2 + sin^2|
+-------------------+------------------+------------------+
| 0.5367821013180484| 30.75534892368792|               1.0|
|  2.528059202813714|144.84712268043324|               1.0|
|  1.813640631048952|103.91395371254823|               1.0|
| 2.9754790311426107|170.48239051414683|0.9999999999999999|
|  0.657423362959178| 37.66758405085815|               1.0|
|0.10747087445354876| 6.157627526768682|               1.0|
|0.36682266433250893| 21.01739049599684|               1.0|
| 1.0211995321679102|  58.5104232339554|               1.0|
|  1.368679228595062| 78.41954330571826|0.9999999999999999|
|  0.873909323189671| 50.07131589590239|               1.0|
+-------------------+------------------+------------------+



/opt/share/spark/python/lib/pyspark.zip/pyspark/sql/functions.py:2051: FutureWarning: Deprecated in 2.1, use degrees instead.


## Combinar DataFrames

![](img/join-types.png)

Podemos distinguir dos manearas de combinar dos `DF`:

* **`union`**: Muy simple, los dos `DF`tienen que tener el mismo número de columnas y se combinan simplemente poniendo uno encima de otro. Similar al `UNION ALL` de *SQL* o al `rbind` de *R*.


* **`join`**: Nos permite cruzar información de dos `DF` por una o más condiciones sobre sus columnas.


Veamos algunos ejemplos de la versatilidad de `join`:


In [39]:
empleados = spark.createDataFrame(
    [
        ("Rafferty", 31),
        ("Jones", 33),
        ("Heisenberg", 33),
        ("Robinson", 34),
        ("Smith", 34),
        ("Williams", None),
    ],
    schema=["LastName", "DepartmentID"],
)

In [40]:
departamentos = spark.createDataFrame(
    [(31, "Sales"), (33, "Engineering"), (34, "Clerical"), (35, "Marketing")],
    schema=["DepartmentID", "DepartmentName"],
)

In [41]:
empleados.show()

+----------+------------+
|  LastName|DepartmentID|
+----------+------------+
|  Rafferty|          31|
|     Jones|          33|
|Heisenberg|          33|
|  Robinson|          34|
|     Smith|          34|
|  Williams|        NULL|
+----------+------------+



In [42]:
departamentos.show()

+------------+--------------+
|DepartmentID|DepartmentName|
+------------+--------------+
|          31|         Sales|
|          33|   Engineering|
|          34|      Clerical|
|          35|     Marketing|
+------------+--------------+



In [43]:
(empleados.join(departamentos, "DepartmentID")).show()

+------------+----------+--------------+
|DepartmentID|  LastName|DepartmentName|
+------------+----------+--------------+
|          31|  Rafferty|         Sales|
|          33|     Jones|   Engineering|
|          33|Heisenberg|   Engineering|
|          34|  Robinson|      Clerical|
|          34|     Smith|      Clerical|
+------------+----------+--------------+



In [44]:
(empleados.join(departamentos, "DepartmentID", "left")).show()

+------------+----------+--------------+
|DepartmentID|  LastName|DepartmentName|
+------------+----------+--------------+
|          31|  Rafferty|         Sales|
|          33|     Jones|   Engineering|
|          33|Heisenberg|   Engineering|
|          34|  Robinson|      Clerical|
|          34|     Smith|      Clerical|
|        NULL|  Williams|          NULL|
+------------+----------+--------------+



In [45]:
(empleados.join(departamentos, "DepartmentID", "full")).show()

+------------+----------+--------------+
|DepartmentID|  LastName|DepartmentName|
+------------+----------+--------------+
|        NULL|  Williams|          NULL|
|          31|  Rafferty|         Sales|
|          33|     Jones|   Engineering|
|          33|Heisenberg|   Engineering|
|          34|  Robinson|      Clerical|
|          34|     Smith|      Clerical|
|          35|      NULL|     Marketing|
+------------+----------+--------------+



In [46]:
(empleados.crossJoin(departamentos)).show()

+----------+------------+------------+--------------+
|  LastName|DepartmentID|DepartmentID|DepartmentName|
+----------+------------+------------+--------------+
|  Rafferty|          31|          31|         Sales|
|  Rafferty|          31|          33|   Engineering|
|     Jones|          33|          31|         Sales|
|     Jones|          33|          33|   Engineering|
|Heisenberg|          33|          31|         Sales|
|Heisenberg|          33|          33|   Engineering|
|  Rafferty|          31|          34|      Clerical|
|  Rafferty|          31|          35|     Marketing|
|     Jones|          33|          34|      Clerical|
|     Jones|          33|          35|     Marketing|
|Heisenberg|          33|          34|      Clerical|
|Heisenberg|          33|          35|     Marketing|
|  Robinson|          34|          31|         Sales|
|  Robinson|          34|          33|   Engineering|
|     Smith|          34|          31|         Sales|
|     Smith|          34|   

¿Qué pasa si hay duplicados en la clave de cruce?

In [47]:
departamentos2 = spark.createDataFrame(
    [
        (31, "Sales"),
        (33, "Engineering"),
        (34, "Clerical"),
        (35, "Marketing"),
        (31, "Ventas"),
    ],
    schema=["DepartmentID", "DepartmentName"],
)

In [48]:
(empleados.join(departamentos2, "DepartmentID")).show()

+------------+----------+--------------+
|DepartmentID|  LastName|DepartmentName|
+------------+----------+--------------+
|          31|  Rafferty|         Sales|
|          31|  Rafferty|        Ventas|
|          33|     Jones|   Engineering|
|          33|Heisenberg|   Engineering|
|          34|  Robinson|      Clerical|
|          34|     Smith|      Clerical|
+------------+----------+--------------+



Por último, veamos como la condición del `join` puede ser mucho más compleja:

In [49]:
productos = spark.createDataFrame(
    [
        ("steak", "1990-01-01", "2000-01-01", 150),
        ("steak", "2000-01-02", "2020-01-01", 180),
        ("fish", "1990-01-01", "2020-01-01", 100),
    ],
    schema=["name", "startDate", "endDate", "price"],
)

In [50]:
pedidos = spark.createDataFrame(
    [("1995-01-01", "steak"), ("2000-01-01", "fish"), ("2005-01-01", "steak")],
    schema=["date", "product"],
)

In [51]:
(
    pedidos.join(
        productos,
        (F.col("product") == F.col("name"))
        & (F.col("date") >= F.col("startDate"))
        & (F.col("date") <= F.col("endDate")),
    )
).show()

+----------+-------+-----+----------+----------+-----+
|      date|product| name| startDate|   endDate|price|
+----------+-------+-----+----------+----------+-----+
|2000-01-01|   fish| fish|1990-01-01|2020-01-01|  100|
|1995-01-01|  steak|steak|1990-01-01|2000-01-01|  150|
|2005-01-01|  steak|steak|2000-01-02|2020-01-01|  180|
+----------+-------+-----+----------+----------+-----+



## Agregaciones

Ya hemos visto el commando `groupBy` que nos sirve para agregar por ciertas columnas. Veremos algunos ejemplos de qué podemos hacer agrupando por columnas:

In [52]:
esquema = T.StructType(
    [
        T.StructField("carat", T.DoubleType()),
        T.StructField("cut", T.StringType()),
        T.StructField("color", T.StringType()),
        T.StructField("clarity", T.StringType()),
        T.StructField("depth", T.DoubleType()),
        T.StructField("table", T.DoubleType()),
        T.StructField("price", T.IntegerType()),
        T.StructField("x", T.DoubleType()),
        T.StructField("y", T.DoubleType()),
        T.StructField("z", T.DoubleType()),
    ]
)

Volvemos a cargar el dataset `diamonds.csv` en este caso ya damos el esquema definido:

In [53]:
diamonds = (spark.read.options(header=True).schema(esquema).csv("/datos/diamonds.csv")).cache()

In [54]:
diamonds.printSchema()

root
 |-- carat: double (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: double (nullable = true)
 |-- table: double (nullable = true)
 |-- price: integer (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: double (nullable = true)



In [55]:
diamonds.groupBy("cut").count().show()

[Stage 85:>                                                         (0 + 1) / 1]

+---------+-----+
|      cut|count|
+---------+-----+
|  Premium|13791|
|    Ideal|21551|
|     Good| 4906|
|     Fair| 1610|
|Very Good|12082|
+---------+-----+



In [56]:
diamonds.groupBy("cut").agg(F.count("*").alias("conteo")).show()

+---------+------+
|      cut|conteo|
+---------+------+
|  Premium| 13791|
|    Ideal| 21551|
|     Good|  4906|
|     Fair|  1610|
|Very Good| 12082|
+---------+------+



In [57]:
diamonds.show(4)

+-----+-------+-----+-------+-----+-----+-----+----+----+----+
|carat|    cut|color|clarity|depth|table|price|   x|   y|   z|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
| 0.23|  Ideal|    E|    SI2| 61.5| 55.0|  326|3.95|3.98|2.43|
| 0.21|Premium|    E|    SI1| 59.8| 61.0|  326|3.89|3.84|2.31|
| 0.23|   Good|    E|    VS1| 56.9| 65.0|  327|4.05|4.07|2.31|
| 0.29|Premium|    I|    VS2| 62.4| 58.0|  334| 4.2|4.23|2.63|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
only showing top 4 rows



Con `agg` Podemos crear varias columnas en la misma agrupación:

In [58]:
agregado = diamonds.groupBy("cut").agg(
    F.count("*").alias("conteo"),
    F.sum("x").alias("x"),
    F.sum("x").alias("y"),
    F.sum("x").alias("z"),
    F.mean("price").alias("media"),
    F.collect_set("clarity").alias("clarity"),
)

In [59]:
agregado.toPandas()

,cut,conteo,x,y,z,media,clarity
0,Premium,13791,82385.88,82385.88,82385.88,4584.257704,"[VS2, SI2, IF, VS1, VVS2, VVS1, I1, SI1]"
1,Ideal,21551,118691.07,118691.07,118691.07,3457.541970,"[VS2, SI2, IF, VS1, VVS2, VVS1, I1, SI1]"
2,Good,4906,28645.08,28645.08,28645.08,3928.864452,"[VS2, SI2, IF, VS1, VVS2, VVS1, I1, SI1]"
3,Fair,1610,10057.50,10057.50,10057.50,4358.757764,"[SI1, SI2, IF, VS1, VVS2, VVS1, I1, VS2]"
4,Very Good,12082,69359.09,69359.09,69359.09,3981.759891,"[VS2, SI2, IF, VS1, VVS2, VVS1, I1, SI1]"


In [60]:
diamonds.unpersist()

DataFrame[carat: double, cut: string, color: string, clarity: string, depth: double, table: double, price: int, x: double, y: double, z: double]

## Funciones Ventanas-Windows

![](img/windows.png)

&nbsp;

Las funciones de tipo [*ventana*](https://en.wikipedia.org/wiki/SQL_window_function) nos permite hacer operaciones analíticas más complejas y usar información de otros registros. Veamos algunos ejemplos:

In [61]:
productRevenue = spark.read.load("/datos/productRevenue.parquet").cache()

In [62]:
productRevenue.count()

10

In [63]:
productRevenue.show()

+----------+----------+-------+
|   product|  category|revenue|
+----------+----------+-------+
|      Thin|Cell phone|   6000|
|    Normal|    Tablet|   1500|
|      Mini|    Tablet|   5500|
|Ultra Thin|Cell phone|   5000|
| Very Thin|Cell phone|   6000|
|       Big|    Tablet|   2500|
|  Bendable|Cell phone|   3000|
|  Foldable|Cell phone|   3000|
|       Pro|    Tablet|   4500|
|      Pro2|    Tablet|   6500|
+----------+----------+-------+



Queremos calcular:

* La difrencia entre el mayor ingreso de la categoría y el actual.
* El puesto (1º,2º,...) a nivel de ingresos para cada categoría.

In [64]:
from pyspark.sql.window import Window

In [65]:
(
    productRevenue.withColumn(
        "revenue_max", F.max("revenue").over(Window.partitionBy("category"))
    ).withColumn("revenue_difference", F.col("revenue_max") - F.col("revenue"))
).show()

+----------+----------+-------+-----------+------------------+
|   product|  category|revenue|revenue_max|revenue_difference|
+----------+----------+-------+-----------+------------------+
|      Thin|Cell phone|   6000|       6000|                 0|
|Ultra Thin|Cell phone|   5000|       6000|              1000|
| Very Thin|Cell phone|   6000|       6000|                 0|
|  Bendable|Cell phone|   3000|       6000|              3000|
|  Foldable|Cell phone|   3000|       6000|              3000|
|    Normal|    Tablet|   1500|       6500|              5000|
|      Mini|    Tablet|   5500|       6500|              1000|
|       Big|    Tablet|   2500|       6500|              4000|
|       Pro|    Tablet|   4500|       6500|              2000|
|      Pro2|    Tablet|   6500|       6500|                 0|
+----------+----------+-------+-----------+------------------+



Además de las funciones que ya hemos visto, también podemos usar las siguientes funciones sobre una ventana:

<table class="table">
<tbody>
<tr>
<td></td>
<td><strong>SQL</strong></td>
<td><strong>DataFrame API</strong></td>
</tr>
<tr>
<td rowspan="5"><strong>Ranking functions</strong></td>
<td>rank</td>
<td>rank</td>
</tr>
<tr>
<td>dense_rank</td>
<td>denseRank</td>
</tr>
<tr>
<td>percent_rank</td>
<td>percentRank</td>
</tr>
<tr>
<td>ntile</td>
<td>ntile</td>
</tr>
<tr>
<td>row_number</td>
<td>rowNumber</td>
</tr>
<tr>
<td rowspan="5"><strong>Analytic functions</strong></td>
<td>cume_dist</td>
<td>cumeDist</td>
</tr>
<tr>
<td>first_value</td>
<td>firstValue</td>
</tr>
<tr>
<td>last_value</td>
<td>lastValue</td>
</tr>
<tr>
<td>lag</td>
<td>lag</td>
</tr>
<tr>
<td>lead</td>
<td>lead</td>
</tr>
</tbody>
</table>

Calculemos ahora el ranqueo pedido:

In [66]:
(
    productRevenue.withColumn(
        "ranking",
        F.rank().over(Window.partitionBy("category").orderBy(F.desc("revenue"))),
    ).withColumn(
        "dense_ranking",
        F.dense_rank().over(Window.partitionBy("category").orderBy(F.desc("revenue"))),
    )
).show()

+----------+----------+-------+-------+-------------+
|   product|  category|revenue|ranking|dense_ranking|
+----------+----------+-------+-------+-------------+
|      Thin|Cell phone|   6000|      1|            1|
| Very Thin|Cell phone|   6000|      1|            1|
|Ultra Thin|Cell phone|   5000|      3|            2|
|  Bendable|Cell phone|   3000|      4|            3|
|  Foldable|Cell phone|   3000|      4|            3|
|      Pro2|    Tablet|   6500|      1|            1|
|      Mini|    Tablet|   5500|      2|            2|
|       Pro|    Tablet|   4500|      3|            3|
|       Big|    Tablet|   2500|      4|            4|
|    Normal|    Tablet|   1500|      5|            5|
+----------+----------+-------+-------+-------------+



Podemos definir la ventana por separado para simplificar el código:

In [67]:
ventana = Window.partitionBy("category").orderBy(F.desc("revenue"))

In [68]:
ranqueados = productRevenue.withColumn("ranking", F.rank().over(ventana)).withColumn(
    "dense_ranking", F.dense_rank().over(ventana)
)

In [69]:
ranqueados.show()

+----------+----------+-------+-------+-------------+
|   product|  category|revenue|ranking|dense_ranking|
+----------+----------+-------+-------+-------------+
|      Thin|Cell phone|   6000|      1|            1|
| Very Thin|Cell phone|   6000|      1|            1|
|Ultra Thin|Cell phone|   5000|      3|            2|
|  Bendable|Cell phone|   3000|      4|            3|
|  Foldable|Cell phone|   3000|      4|            3|
|      Pro2|    Tablet|   6500|      1|            1|
|      Mini|    Tablet|   5500|      2|            2|
|       Pro|    Tablet|   4500|      3|            3|
|       Big|    Tablet|   2500|      4|            4|
|    Normal|    Tablet|   1500|      5|            5|
+----------+----------+-------+-------+-------------+



## Data Sources

![](img/blog-illustration-01.png)

Desde spark podemos leer todo tipo de formatos, veamos algunos ejemplos más avanzados. Lo primero de todo vamos a cambiar cada uno a la databse de hive de nuestro usuario:

In [70]:
mi_user = os.environ.get("USER")
print(mi_user)

jayuso


In [71]:
spark.catalog.setCurrentDatabase(mi_user)

Guardamos los datos en Hive:

In [72]:
ranqueados.write.saveAsTable("ranqueos")

In [73]:
pd.DataFrame(spark.catalog.listTables())

,name,catalog,namespace,description,tableType,isTemporary
0,agregado_2,spark_catalog,[jayuso],None,MANAGED,False
1,audiencias,spark_catalog,[jayuso],None,MANAGED,False
2,audiencias_large,spark_catalog,[jayuso],None,MANAGED,False
3,dns_agregado,spark_catalog,[jayuso],None,MANAGED,False
4,huge_table,spark_catalog,[jayuso],None,MANAGED,False
5,huge_table_v2,spark_catalog,[jayuso],None,MANAGED,False
6,passenger_role_type,spark_catalog,[jayuso],None,MANAGED,False
7,prueba,spark_catalog,[jayuso],None,MANAGED,False
8,prueba5,spark_catalog,[jayuso],None,EXTERNAL,False
9,ranqueos,spark_catalog,[jayuso],None,MANAGED,False


También podemos borrar la tabla creada con sentencia *SQL*:

In [74]:
spark.sql(" DROP TABLE ranqueos")

DataFrame[]

In [75]:
pd.DataFrame(spark.catalog.listTables())

,name,catalog,namespace,description,tableType,isTemporary
0,agregado_2,spark_catalog,[jayuso],None,MANAGED,False
1,audiencias,spark_catalog,[jayuso],None,MANAGED,False
2,audiencias_large,spark_catalog,[jayuso],None,MANAGED,False
3,dns_agregado,spark_catalog,[jayuso],None,MANAGED,False
4,huge_table,spark_catalog,[jayuso],None,MANAGED,False
5,huge_table_v2,spark_catalog,[jayuso],None,MANAGED,False
6,passenger_role_type,spark_catalog,[jayuso],None,MANAGED,False
7,prueba,spark_catalog,[jayuso],None,MANAGED,False
8,prueba5,spark_catalog,[jayuso],None,EXTERNAL,False
9,ratings,spark_catalog,[jayuso],None,MANAGED,False


Además podemos guardar la tabla directamen en un direcotrio del hdfs:

In [76]:
ranqueados.write.parquet("ranqueos.parquet")

En este caso tanto `parquet` como `save` hace lo mismo guardar los datos en formato parquet. La última sentencia nos ha dado error porque ya exitía el archivo.

### Save Modes
Por defecto al guardar un `DF` obtenemos error si existe el directorio, podemos configurar para que esto no ocurra:

<table class="table">
<tbody><tr><th>Scala/Java</th><th>Any Language</th><th>Meaning</th></tr>
<tr>
  <td><code>SaveMode.ErrorIfExists</code> (default)</td>
  <td><code>"error"</code> (default)</td>
  <td>
    When saving a DataFrame to a data source, if data already exists,
    an exception is expected to be thrown.
  </td>
</tr>
<tr>
  <td><code>SaveMode.Append</code></td>
  <td><code>"append"</code></td>
  <td>
    When saving a DataFrame to a data source, if data/table already exists,
    contents of the DataFrame are expected to be appended to existing data.
  </td>
</tr>
<tr>
  <td><code>SaveMode.Overwrite</code></td>
  <td><code>"overwrite"</code></td>
  <td>
    Overwrite mode means that when saving a DataFrame to a data source,
    if data/table already exists, existing data is expected to be overwritten by the contents of
    the DataFrame.
  </td>
</tr>
<tr>
  <td><code>SaveMode.Ignore</code></td>
  <td><code>"ignore"</code></td>
  <td>
    Ignore mode means that when saving a DataFrame to a data source, if data already exists,
    the save operation is expected to not save the contents of the DataFrame and to not
    change the existing data. This is similar to a <code>CREATE TABLE IF NOT EXISTS</code> in SQL.
  </td>
</tr>
</tbody></table>

In [77]:
ranqueados.write.mode("overwrite").save("ranqueos.parquet")

In [78]:
spark.read.load("ranqueos.parquet").count()

10

Con `append` añadimos a la tabla existente:

In [79]:
for i in range(3):
    ranqueados.write.mode("append").save("ranqueos.parquet")

In [80]:
spark.read.load("ranqueos.parquet").count()

40

### JSON

![](img/json.png)

También podemos leer y escribir en formato JSON, aunque realmente se usa el formato [*JSON lines*](http://jsonlines.org/) dónde cada línea es un registro:

In [81]:
!hadoop fs -text /datos/pelis.json | head -n 3

{"ratingvalue":9.0,"titulo":"El padrino: Parte II"}
{"ratingvalue":8.9,"titulo":"Pulp Fiction"}
{"ratingvalue":8.9,"titulo":"12 hombres sin piedad"}


In [82]:
pelis = spark.read.json("/datos/pelis.json")

In [83]:
pelis.printSchema()

root
 |-- ratingvalue: double (nullable = true)
 |-- titulo: string (nullable = true)



In [84]:
pelis.limit(10).toPandas()

,ratingvalue,titulo
0,9.0,El padrino: Parte II
1,8.9,Pulp Fiction
2,8.9,12 hombres sin piedad
3,8.9,La lista de Schindler
4,9.0,El caballero oscuro
5,9.2,El padrino
6,8.9,El señor de los anillos: El retorno del rey
7,8.9,"El bueno, el feo y el malo"
8,8.8,El club de la lucha
9,9.3,Cadena perpetua


### JDBC (Java Database Connectivity)

![](img/java_jdbc.png)

JDBC es un protocolo altamente utilizado por las bases de datos. De esta manera, muchas base de dato tiene un *driver* para este protocolo. Veamos por ejemplo como conectarnos a *MySQL*

**NOTA:** Observar como al iniciar la sesión de spark pusimos:



```python
.set("spark.jars","/var/lib/sqoop/mysql-connector-java-5.1.44-bin.jar")
```
Para añadir el driver a la sesión.


In [85]:
movies = (
    spark.read.format("jdbc")
    .option("url", "jdbc:mysql://manager01.bigdata.alumnos.upcont.es/movielens")
    .option("driver", "com.mysql.jdbc.Driver")
    .option("dbtable", "movies")
    .option("user", "root")
    .option("password", "root")
    .option("useSSL", False)
    .load()
)

In [86]:
movies.count()

9125

In [87]:
movies.show()

+--------+--------------------+--------------------+
|movie_id|               title|              genres|
+--------+--------------------+--------------------+
|       1|    Toy Story (1995)|Adventure|Animati...|
|       2|      Jumanji (1995)|Adventure|Childre...|
|       3|Grumpier Old Men ...|    Comedy|Romance\r|
|       4|Waiting to Exhale...|Comedy|Drama|Roma...|
|       5|Father of the Bri...|            Comedy\r|
|       6|         Heat (1995)|Action|Crime|Thri...|
|       7|      Sabrina (1995)|    Comedy|Romance\r|
|       8| Tom and Huck (1995)|Adventure|Children\r|
|       9| Sudden Death (1995)|            Action\r|
|      10|    GoldenEye (1995)|Action|Adventure|...|
|      11| "American President|         The (1995)"|
|      12|Dracula: Dead and...|     Comedy|Horror\r|
|      13|        Balto (1995)|Adventure|Animati...|
|      14|        Nixon (1995)|             Drama\r|
|      15|Cutthroat Island ...|Action|Adventure|...|
|      16|       Casino (1995)|       Crime|Dr

Arreglamos el campo `genres`:

In [88]:
movies.take(4)

[Row(movie_id=1, title='Toy Story (1995)', genres='Adventure|Animation|Children|Comedy|Fantasy\r'),
 Row(movie_id=2, title='Jumanji (1995)', genres='Adventure|Children|Fantasy\r'),
 Row(movie_id=3, title='Grumpier Old Men (1995)', genres='Comedy|Romance\r'),
 Row(movie_id=4, title='Waiting to Exhale (1995)', genres='Comedy|Drama|Romance\r')]

In [89]:
movies = movies.withColumn("genres", F.regexp_replace("genres", r"\r$", ""))

In [90]:
movies.take(4)

[Row(movie_id=1, title='Toy Story (1995)', genres='Adventure|Animation|Children|Comedy|Fantasy'),
 Row(movie_id=2, title='Jumanji (1995)', genres='Adventure|Children|Fantasy'),
 Row(movie_id=3, title='Grumpier Old Men (1995)', genres='Comedy|Romance'),
 Row(movie_id=4, title='Waiting to Exhale (1995)', genres='Comedy|Drama|Romance')]

In [91]:
tags = (
    movies.withColumn("tag", F.split("genres", r"\|"))
    .drop("genres")
    .withColumn("tag", F.explode_outer("tag"))
)

In [92]:
tags.show()

+--------+--------------------+---------+
|movie_id|               title|      tag|
+--------+--------------------+---------+
|       1|    Toy Story (1995)|Adventure|
|       1|    Toy Story (1995)|Animation|
|       1|    Toy Story (1995)| Children|
|       1|    Toy Story (1995)|   Comedy|
|       1|    Toy Story (1995)|  Fantasy|
|       2|      Jumanji (1995)|Adventure|
|       2|      Jumanji (1995)| Children|
|       2|      Jumanji (1995)|  Fantasy|
|       3|Grumpier Old Men ...|   Comedy|
|       3|Grumpier Old Men ...|  Romance|
|       4|Waiting to Exhale...|   Comedy|
|       4|Waiting to Exhale...|    Drama|
|       4|Waiting to Exhale...|  Romance|
|       5|Father of the Bri...|   Comedy|
|       6|         Heat (1995)|   Action|
|       6|         Heat (1995)|    Crime|
|       6|         Heat (1995)| Thriller|
|       7|      Sabrina (1995)|   Comedy|
|       7|      Sabrina (1995)|  Romance|
|       8| Tom and Huck (1995)|Adventure|
+--------+--------------------+---

In [93]:
tags.groupBy("tag").count().filter("count>2").orderBy(F.desc("count")).show()

+------------+-----+
|         tag|count|
+------------+-----+
|       Drama| 3264|
|      Comedy| 2611|
|    Thriller| 1305|
|      Action| 1240|
|     Romance| 1200|
|   Adventure|  856|
|       Crime|  853|
|      Horror|  657|
|      Sci-Fi|  656|
|     Fantasy|  492|
|    Children|  425|
|     Mystery|  393|
| Documentary|  389|
|   Animation|  353|
|     Musical|  306|
|         War|  278|
|        IMAX|  128|
|     Western|  120|
|   Film-Noir|   93|
| The (1999)"|   48|
+------------+-----+
only showing top 20 rows



In [94]:
resumen_pelis = (
    tags.groupBy("movie_id")
    .agg(F.collect_set("tag").alias("tags"))
    .withColumn("conteo", F.size("tags"))
)

In [95]:
resumen_pelis.show()

+--------+--------------------+------+
|movie_id|                tags|conteo|
+--------+--------------------+------+
|       1|[Fantasy, Adventu...|     5|
|       2|[Fantasy, Adventu...|     3|
|       3|   [Comedy, Romance]|     2|
|       4|[Comedy, Drama, R...|     3|
|       5|            [Comedy]|     1|
|       6|[Thriller, Action...|     3|
|       7|   [Comedy, Romance]|     2|
|       8|[Adventure, Child...|     2|
|       9|            [Action]|     1|
|      10|[Thriller, Action...|     3|
|      11|      [ The (1995)"]|     1|
|      12|    [Comedy, Horror]|     2|
|      13|[Adventure, Child...|     3|
|      14|             [Drama]|     1|
|      15|[Action, Adventur...|     3|
|      16|      [Crime, Drama]|     2|
|      17|    [Drama, Romance]|     2|
|      18|            [Comedy]|     1|
|      19|            [Comedy]|     1|
|      20|[Thriller, Action...|     5|
+--------+--------------------+------+
only showing top 20 rows



Podemos escribir la información procesada del *MySQL* directamente a un fichero *JSON* por ejemplo:

In [96]:
resumen_pelis.write.mode("overwrite").json("pelis.json")

In [97]:
for i in spark.sparkContext.textFile("pelis.json").take(4):
    print(i)

{"movie_id":1,"tags":["Fantasy","Adventure","Comedy","Children","Animation"],"conteo":5}
{"movie_id":2,"tags":["Fantasy","Adventure","Children"],"conteo":3}
{"movie_id":3,"tags":["Comedy","Romance"],"conteo":2}
{"movie_id":4,"tags":["Comedy","Drama","Romance"],"conteo":3}


In [98]:
spark.read.json("pelis.json").show()

+------+--------+--------------------+
|conteo|movie_id|                tags|
+------+--------+--------------------+
|     5|       1|[Fantasy, Adventu...|
|     3|       2|[Fantasy, Adventu...|
|     2|       3|   [Comedy, Romance]|
|     3|       4|[Comedy, Drama, R...|
|     1|       5|            [Comedy]|
|     3|       6|[Thriller, Action...|
|     2|       7|   [Comedy, Romance]|
|     2|       8|[Adventure, Child...|
|     1|       9|            [Action]|
|     3|      10|[Thriller, Action...|
|     1|      11|      [ The (1995)"]|
|     2|      12|    [Comedy, Horror]|
|     3|      13|[Adventure, Child...|
|     1|      14|             [Drama]|
|     3|      15|[Action, Adventur...|
|     2|      16|      [Crime, Drama]|
|     2|      17|    [Drama, Romance]|
|     1|      18|            [Comedy]|
|     1|      19|            [Comedy]|
|     5|      20|[Thriller, Action...|
+------+--------+--------------------+
only showing top 20 rows



In [99]:
spark.stop()